# Build Dimension Product
1. read the silver products table
2. create the product surrogate key
3. select the required columns
4. write the transfomred data to gold dim_products table

In [0]:
#Imports
from pyspark.sql.functions import col,row_number
from pyspark.sql import Window

### Step1 - read the silver products table

In [0]:
products_df = spark.read.table("olist_catalog.silver.products")

### Step2 - create the product surrogate key

In [0]:
window_spec=Window.orderBy("product_id")
dim_products_df = products_df.withColumn("product_sk", row_number().over(window_spec))

### Step3 - select the required columns

In [0]:
dim_products_df = (
    dim_products_df.select(
        "product_sk",
        "product_id",
        "product_category_name",
        "product_weight_g",
        "product_length_cm",
        "product_width_cm",
        "product_height_cm",
        "product_photos_qty"
    )
)

### Step4 - write the transfomred data to gold dim_products table

In [0]:
(
    dim_products_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("olist_catalog.gold.dim_products")
)

In [0]:
%sql
select * from olist_catalog.gold.dim_products